In [ ]:
# assign these paths for inference
from pathlib import Path

SAGITTAL_DIR = Path(r"")
AXIAL_DIR = Path(r"")
CORONAL_DIR = Path(r"")
ATLAS_PATH = Path(r"")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import random, urllib.request
import time
import cv2
import warnings
from scipy.spatial import ConvexHull
import tifffile
import SimpleITK as sitk
from scipy.ndimage import binary_fill_holes
from skimage.filters import threshold_li, threshold_otsu
from skimage.morphology import closing, disk
from skimage.measure import label, regionprops
from skimage.transform import resize
from skimage.metrics import hausdorff_distance
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=Image.DecompressionBombWarning)

print("Imports OK.")

In [ ]:
AXIS_INPUT_DIRS = {
    "sagittal": Path(SAGITTAL_DIR) if SAGITTAL_DIR else None,
    "axial": Path(AXIAL_DIR) if AXIAL_DIR else None,
    "coronal": Path(CORONAL_DIR) if CORONAL_DIR else None,
}

SAMPLES_PER_DATASET = 12
SUPPORTED = {".png", ".jpg", ".jpeg", ".jfif", ".bmp", ".tif", ".tiff", ".czi"}

PROCESS_SIZE = 256

AXIS_SLICE_SEARCH = {
    "coronal":  (0.05, 0.95, 15),
    "sagittal": (0.05, 0.95, 15),
    "axial":    (0.05, 0.95, 25),
}
AXIS_DIM = {"sagittal": 2, "coronal": 0, "axial": 1}

if not ATLAS_PATH.exists():
    print("Downloading Allen Mouse Atlas (~500 MB)…")
    urllib.request.urlretrieve(
        "https://download.alleninstitute.org/informatics-archive/"
        "current-release/mouse_ccf/average_template_25.nrrd",
        ATLAS_PATH)
    print("Done.")
else:
    print("Found existing atlas.")

_sitk = sitk.ReadImage(str(ATLAS_PATH))
_ori = sitk.DICOMOrientImageFilter()
_ori.SetDesiredCoordinateOrientation("RAS")
_sitk = _ori.Execute(_sitk)
_sitk = sitk.PermuteAxes(_sitk, [2, 1, 0])
atlas_vol = sitk.GetArrayFromImage(_sitk).astype(np.float32)
print(f"Atlas shape: {atlas_vol.shape}  (coronal × axial × sagittal)")


def _resize_to(arr: np.ndarray, size: int) -> np.ndarray:
    """Resize longest edge to `size`, preserving aspect ratio."""
    h, w = arr.shape
    scale = size / max(h, w)
    new_h, new_w = max(1, int(h * scale)), max(1, int(w * scale))
    return resize(arr, (new_h, new_w), anti_aliasing=True, order=1).astype(np.float32)


def get_atlas_candidates(axis: str) -> list[np.ndarray]:
    start, end, n = AXIS_SLICE_SEARCH[axis]
    dim = AXIS_DIM[axis]
    fracs = np.linspace(start, end, n)
    slices = []
    for frac in fracs:
        idx = int(atlas_vol.shape[dim] * frac)
        sl = atlas_vol[idx] if dim == 0 else (atlas_vol[:, idx] if dim == 1 else atlas_vol[:, :, idx])
        if axis == "sagittal":
            sl = np.rot90(sl, k=1)
            sl = np.fliplr(sl)
        else:
            sl = np.rot90(sl, k=2)
        sl = _resize_to(sl.astype(np.float32), PROCESS_SIZE)
        slices.append(sl)
    return slices


ATLAS_CANDIDATES = {axis: get_atlas_candidates(axis) for axis in AXIS_DIM}
ATLAS_RAW = {axis: ATLAS_CANDIDATES[axis][len(ATLAS_CANDIDATES[axis]) // 2] for axis in AXIS_DIM}

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for i, axis in enumerate(["coronal", "sagittal", "axial"]):
    sl = ATLAS_RAW[axis]
    axs[i].imshow((sl - sl.min()) / (sl.max() - sl.min() + 1e-8), cmap="gray")
    axs[i].set_title(f"Atlas — {axis}"); axs[i].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
def normalize(arr: np.ndarray) -> np.ndarray:
    lo, hi = float(arr.min()), float(arr.max())
    return (arr - lo) / (hi - lo + 1e-8)


def load_slice(path) -> np.ndarray:
    path = Path(path); ext = path.suffix.lower()
    if ext in (".png", ".jpg", ".jpeg", ".jfif", ".bmp"):
        img = Image.open(path)
        if ext in (".jpg", ".jpeg"):
            img.draft("L", (PROCESS_SIZE * 2, PROCESS_SIZE * 2))
        arr = np.array(img.convert("L"), dtype=np.float32)
    elif ext in (".tif", ".tiff"):
        arr = tifffile.imread(str(path)).astype(np.float32)
        if arr.ndim == 3:
            arr = arr.mean(2) if arr.shape[2] <= 4 else arr[arr.shape[0] // 2]
    elif ext == ".czi":
        import czifile
        arr = czifile.imread(str(path)).squeeze().astype(np.float32)
        if arr.ndim == 3:
            arr = arr.mean(2) if arr.shape[2] <= 4 else arr[arr.shape[0] // 2]
    else:
        raise ValueError(f"Unsupported format: {ext}")

    return _resize_to(arr, PROCESS_SIZE)


def is_valid_slice(path) -> bool:
    try:
        ext = Path(path).suffix.lower()
        if ext in (".png", ".jpg", ".jpeg", ".jfif", ".bmp"):
            with Image.open(path) as img:
                w, h = img.size
                return min(h, w) >= 64
        elif ext in (".tif", ".tiff"):
            with tifffile.TiffFile(str(path)) as tif:
                h, w = tif.pages[0].shape[:2]
                return min(h, w) >= 64
        return True
    except Exception:
        return False

In [ ]:
def detect_polarity(img: np.ndarray) -> int:
    n = normalize(img); h, w = n.shape
    s = max(h//8, w//8, 10)
    border = np.concatenate([n[:s,:s].ravel(), n[:s,-s:].ravel(),
                              n[-s:,:s].ravel(), n[-s:,-s:].ravel()])
    return 1 if np.median(n[h//4:3*h//4, w//4:3*w//4]) >= np.median(border) else -1


def get_tissue_mask(img: np.ndarray, min_area_frac: float = 0.02) -> np.ndarray:
    img_n = normalize(img)
    img_w = img_n if detect_polarity(img_n) == 1 else (1.0 - img_n)
    try:
        thresh = float(threshold_li(img_w))
    except:
        thresh = float(threshold_otsu(img_w))
    mask = closing(img_w > thresh, disk(5))
    mask = binary_fill_holes(mask)
    labeled = label(mask)
    if labeled.max() == 0:
        return mask.astype(bool)
    props = regionprops(labeled)
    min_area = max(int(img_n.size * min_area_frac), 200)
    big = sorted([p for p in props if p.area > min_area], key=lambda p: -p.area)
    if not big:
        big = [max(props, key=lambda p: p.area)]
    clean = np.zeros(mask.shape, bool)
    for p in big[:3]:
        clean[labeled == p.label] = True
    return clean


def main_body_mask(mask: np.ndarray) -> np.ndarray:
    labeled = label(mask)
    if labeled.max() == 0:
        return mask.astype(bool)
    biggest = max(regionprops(labeled), key=lambda p: p.area)
    return labeled == biggest.label


In [ ]:
def _ncc(a: np.ndarray, b: np.ndarray) -> float:
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / (np.sqrt((a ** 2).sum() * (b ** 2).sum()) + 1e-8))


def _ncc_polarity_aware(inp: np.ndarray, atlas: np.ndarray) -> float:
    inp_r = normalize(resize(inp.astype(np.float32), atlas.shape, anti_aliasing=True, order=1))
    atlas_n = normalize(atlas.astype(np.float32))
    if detect_polarity(inp) != detect_polarity(atlas_n):
        inp_r = 1.0 - inp_r
    return _ncc(inp_r, atlas_n)


def principal_axes_angle(mask: np.ndarray) -> float:
    coords = np.column_stack(np.where(mask))
    if len(coords) < 50:
        return 0.0
    pca = PCA(n_components=2).fit(coords)
    vec = pca.components_[0]
    return np.degrees(np.arctan2(-vec[0], vec[1]))


def best_rotation(img: np.ndarray, atlas_slice: np.ndarray):
    atlas_mask = get_tissue_mask(atlas_slice)
    target_angle = principal_axes_angle(atlas_mask)

    img_mask = get_tissue_mask(img)
    brain_angle = principal_axes_angle(img_mask)

    best_score, best_k = -np.inf, 0
    nccs = []
    for k in range(4):
        candidate = np.rot90(img, k=k)
        ncc_score = _ncc_polarity_aware(candidate, atlas_slice)
        nccs.append(ncc_score)
        cand_mask = np.rot90(img_mask, k=k)
        cand_angle = principal_axes_angle(cand_mask)
        residual = (cand_angle - target_angle + 180) % 360 - 180
        pca_score = np.cos(np.radians(residual))
        combined = 0.7 * pca_score + 0.3 * ncc_score
        if combined > best_score:
            best_score, best_k = combined, k

    sorted_nccs = sorted(nccs, reverse=True)
    margin = float(sorted_nccs[0] - sorted_nccs[1])

    rotated = np.rot90(img, k=best_k)
    return rotated, best_k, float(best_k * 90), float(nccs[best_k]), margin


def resolve_flip(img: np.ndarray, atlas_slice: np.ndarray):
    candidates = {"none": img, "lr": np.fliplr(img), "ud": np.flipud(img), "both": np.flipud(np.fliplr(img))}
    best_ncc, best_key = -np.inf, "none"
    for key, cand in candidates.items():
        score = _ncc_polarity_aware(cand, atlas_slice)
        if score > best_ncc:
            best_ncc, best_key = score, key
    return candidates[best_key], (best_key if best_key != "none" else None)


def find_best_atlas_slice(img: np.ndarray, axis: str) -> np.ndarray:
    candidates = ATLAS_CANDIDATES[axis]

    fast_scores = [(i, _ncc_polarity_aware(img, sl)) for i, sl in enumerate(candidates)]
    fast_scores.sort(key=lambda x: -x[1])
    top_indices = [i for i, _ in fast_scores[:3]]

    best_ncc, best_sl = -np.inf, ATLAS_RAW[axis]
    for i in top_indices:
        sl = candidates[i]
        ncc = max(_ncc_polarity_aware(np.rot90(img, k=k), sl) for k in range(4))
        if ncc > best_ncc:
            best_ncc, best_sl = ncc, sl
    return best_sl


def randomize_orientation(img: np.ndarray, rng: np.random.Generator | None = None):
    rng = rng or np.random.default_rng()
    k = int(rng.integers(0, 4))
    out = np.rot90(img, k=k)
    flip_lr = bool(rng.random() < 0.5)
    flip_ud = bool(rng.random() < 0.5)
    if flip_lr:
        out = np.fliplr(out)
    if flip_ud:
        out = np.flipud(out)
    return out, {"k": k, "flip_lr": flip_lr, "flip_ud": flip_ud}

In [ ]:
def _mask_axis_extents(mask):
    coords = np.column_stack(np.where(mask))
    pca = PCA(n_components=2).fit(coords)
    proj = pca.transform(coords)
    main_extent = proj[:, 0].max() - proj[:, 0].min()
    short_extent = proj[:, 1].max() - proj[:, 1].min()
    return main_extent, short_extent

def classify_slice_type(mask):
    main_ext, short_ext = _mask_axis_extents(mask)
    aspect = short_ext / (main_ext + 1e-8)
    return ("circle" if aspect > 0.62 else "elongated"), aspect

def _radial_symmetry_map(mask, intensity, radii, mag_percentile=60, kappa=8.0, alpha=2.0):
    h, w = mask.shape
    img8 = (normalize(intensity) * 255).astype(np.float32)
    blur = cv2.GaussianBlur(img8, (3, 3), 1.0)

    gx = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    mag_norm = mag / (mag.max() + 1e-6)
    with np.errstate(invalid="ignore", divide="ignore"):
        gx_n = np.where(mag > 1e-6, gx / (mag + 1e-6), 0.0)
        gy_n = np.where(mag > 1e-6, gy / (mag + 1e-6), 0.0)

    strong = mag_norm > np.percentile(mag_norm, mag_percentile)
    ys, xs = np.where(strong)

    S = np.zeros((h, w), np.float32)
    for r in radii:
        O = np.zeros((h, w), np.float32)
        M = np.zeros((h, w), np.float32)

        px = np.round(xs + gx_n[ys, xs] * r).astype(int)
        py = np.round(ys + gy_n[ys, xs] * r).astype(int)
        v = (px >= 0) & (px < w) & (py >= 0) & (py < h)
        np.add.at(O, (py[v], px[v]), 1.0)
        np.add.at(M, (py[v], px[v]), mag_norm[ys[v], xs[v]])

        nx = np.round(xs - gx_n[ys, xs] * r).astype(int)
        ny = np.round(ys - gy_n[ys, xs] * r).astype(int)
        vn = (nx >= 0) & (nx < w) & (ny >= 0) & (ny < h)
        np.add.at(O, (ny[vn], nx[vn]), -1.0)
        np.add.at(M, (ny[vn], nx[vn]), mag_norm[ys[vn], xs[vn]])

        O_clip = np.clip(np.abs(O), 0, kappa) / kappa
        F_r = M * (O_clip ** alpha)
        F_r = cv2.GaussianBlur(F_r, (0, 0), max(1.0, r / 4))
        S += F_r

    return S / max(1, len(radii))

def _ring_count_at(mask, mag, cx, cy, radii_scan, mag_hi):
    from scipy.signal import find_peaks
    h, w = mask.shape
    thetas = np.linspace(0, 2 * np.pi, 48, endpoint=False)
    cos_t, sin_t = np.cos(thetas), np.sin(thetas)
    profile = np.empty(len(radii_scan))
    for i, r in enumerate(radii_scan):
        xr = np.round(cx + r * cos_t).astype(int)
        yr = np.round(cy + r * sin_t).astype(int)
        valid = (xr >= 0) & (xr < w) & (yr >= 0) & (yr < h)
        profile[i] = 0.0 if not valid.any() else (mag[yr[valid], xr[valid]] > mag_hi).mean()
    peaks, props = find_peaks(profile, prominence=0.12, distance=2)
    return len(peaks), profile, peaks

def _fine_graininess_map(intensity):
    img8 = (normalize(intensity) * 255).astype(np.uint8)
    fine = cv2.Laplacian(img8, cv2.CV_32F, ksize=1)
    return cv2.GaussianBlur(np.abs(fine), (3, 3), 0)

def _find_internal_circle(mask, intensity, min_margin_frac=0.06, radius_step=3):
    from skimage.feature import peak_local_max

    h, w = mask.shape
    minr = max(4, int(0.08 * min(h, w)))
    maxr = int(0.24 * min(h, w))
    if maxr <= minr:
        return False, None, None, 0.0
    frst_radii = np.linspace(minr, maxr, 5).astype(int)

    S = _radial_symmetry_map(mask, intensity, frst_radii)

    margin_px = max(3, int(min_margin_frac * min(h, w)))
    k = margin_px * 2 + 1
    interior = cv2.erode(mask.astype(np.uint8), np.ones((k, k), np.uint8)).astype(bool)
    S_masked = np.where(interior, S, 0.0)
    if S_masked.max() <= 0:
        return False, None, None, 0.0

    coords = peak_local_max(S_masked, min_distance=max(3, minr // 2), num_peaks=8,
                             threshold_rel=0.3)
    if len(coords) == 0:
        return False, None, None, 0.0

    img8 = (normalize(intensity) * 255).astype(np.uint8)
    blur = cv2.GaussianBlur(img8, (3, 3), 1.0)
    gx = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)
    mag_hi = np.percentile(mag[mask], 60) if mask.any() else np.percentile(mag, 60)
    radii_scan = np.arange(minr, maxr, radius_step)

    grain = _fine_graininess_map(intensity)
    tissue_grain_mean = grain[mask].mean() + 1e-6
    yy, xx = np.mgrid[0:h, 0:w]
    r_eval = max(3, int(0.5 * (minr + maxr) / 2))

    best = None
    for cy_, cx_ in coords:
        n_rings, _, _ = _ring_count_at(mask, mag, cx_, cy_, radii_scan, mag_hi)
        frst_score = float(S_masked[cy_, cx_])

        disk = (xx - cx_) ** 2 + (yy - cy_) ** 2 <= r_eval ** 2
        grain_ratio = (grain[disk].mean() / tissue_grain_mean) if disk.any() else 1.0
        grain_penalty = 1.0 + max(0.0, grain_ratio - 1.0)  # >1 only if grainier than typical tissue

        score = frst_score * (1.0 + 0.5 * n_rings) / grain_penalty
        if best is None or score > best[0]:
            best = (score, cx_ / w, cy_ / h)

    if best is None:
        return False, None, None, 0.0
    score, cxn, cyn = best
    return True, cxn, cyn, score

def _mask_bbox_hw(mask):
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return 0, 0
    h = int(ys.max() - ys.min() + 1)
    w = int(xs.max() - xs.min() + 1)
    return h, w

def _quadrant(pt, h, w):
    y, x = pt
    vert = "top" if y < h / 2 else "bottom"
    horiz = "left" if x < w / 2 else "right"
    return vert, horiz

# --- elongated-only pipeline (from elongated_slices_final.ipynb) -----------

def _split_tissue_by_neck(mask, min_distance_frac=0.12, min_area_frac=0.05):
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed

    h, w = mask.shape
    dist = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 5)

    min_dist_px = max(5, int(round(min_distance_frac * min(h, w))))
    coords = peak_local_max(dist, min_distance=min_dist_px,
                             labels=mask.astype(int), exclude_border=False)
    if len(coords) < 2:
        return None, None, None, None

    markers = np.zeros(mask.shape, dtype=np.int32)
    for i, (y, x) in enumerate(coords):
        markers[y, x] = i + 1
    labels_ws = watershed(-dist, markers, mask=mask)

    region_ids = [i for i in np.unique(labels_ws) if i != 0]
    areas = {i: int((labels_ws == i).sum()) for i in region_ids}
    largest_id = max(areas, key=areas.get)
    total_area = float(mask.sum())

    coords_all = np.column_stack(np.where(mask))
    pca = PCA(n_components=1).fit(coords_all)
    largest_ys, largest_xs = np.where(labels_ws == largest_id)
    largest_proj = pca.transform([[largest_ys.mean(), largest_xs.mean()]])[0, 0]

    best = None  # (axis_dist, label)
    for i in region_ids:
        if i == largest_id or areas[i] < min_area_frac * total_area:
            continue
        ys, xs = np.where(labels_ws == i)
        centroid = (float(ys.mean()), float(xs.mean()))
        proj = pca.transform([centroid])[0, 0]
        axis_dist = abs(proj - largest_proj)
        if best is None or axis_dist > best[0]:
            best = (axis_dist, i)

    cb_id = best[1] if best is not None else None
    main_id = largest_id

    if cb_id is not None:
        merge_sel = (labels_ws != cb_id) & (labels_ws != 0)
        labels_ws = np.where(merge_sel, main_id, labels_ws).astype(labels_ws.dtype)
        areas = {main_id: int(merge_sel.sum()), cb_id: int(areas[cb_id])}

    return labels_ws, main_id, cb_id, areas

def _locate_cerebellum_via_separation(mask, min_distance_frac=0.12, min_area_frac=0.05):
    labels_ws, main_id, cb_id, areas = _split_tissue_by_neck(mask, min_distance_frac, min_area_frac)
    if labels_ws is None or cb_id is None:
        return False, None, 0.0, None

    ys, xs = np.where(labels_ws == cb_id)
    centroid = (float(ys.mean()), float(xs.mean()))
    quad = _quadrant(centroid, *mask.shape)
    return True, centroid, float(areas[cb_id]), quad

def _texture_energy_map(intensity):
    img8 = (normalize(intensity) * 255).astype(np.uint8)
    lap = cv2.Laplacian(img8, cv2.CV_32F, ksize=3)
    return cv2.GaussianBlur(np.abs(lap), (9, 9), 0)

def _mask_ends_along_axis(mask, pct=12):
    coords = np.column_stack(np.where(mask))
    if len(coords) < 50:
        return None
    pca = PCA(n_components=1).fit(coords)
    proj = pca.transform(coords).ravel()
    lo_thr, hi_thr = np.percentile(proj, [pct, 100 - pct])
    end_lo = coords[proj <= lo_thr]
    end_hi = coords[proj >= hi_thr]
    return end_lo, end_hi

def _locate_cerebellum_via_texture(mask, intensity):
    ends = _mask_ends_along_axis(mask)
    if ends is None:
        return False, None, None
    end_a, end_b = ends
    energy = _texture_energy_map(intensity)
    e_a = energy[end_a[:, 0], end_a[:, 1]].mean()
    e_b = energy[end_b[:, 0], end_b[:, 1]].mean()
    cb_pts = end_a if e_a >= e_b else end_b
    centroid = cb_pts.mean(axis=0)
    quad = _quadrant(centroid, *mask.shape)
    return True, quad, centroid

def _pca_vertical_width_axes_elongated(mask):
    coords = np.column_stack(np.where(mask)).astype(np.float64)  # (y, x)
    centroid = coords.mean(axis=0)
    pca = PCA(n_components=2).fit(coords)
    comps = pca.components_  # each row is a unit vector in (y, x) space
    v_idx = int(np.argmax(np.abs(comps[:, 0])))   # more "row-aligned" axis
    u_idx = 1 - v_idx
    return centroid, comps[v_idx], comps[u_idx]

def _half_widths_by_pca_elongated(mask, n_bins=16):
    ys, xs = np.where(mask)
    if len(ys) < 20:
        return 0.0, 0.0, None

    coords = np.column_stack([ys, xs]).astype(np.float64)
    centroid, v, u = _pca_vertical_width_axes_elongated(mask)
    rel = coords - centroid
    t = rel @ v   # position along the tilt-corrected top-to-bottom axis
    s = rel @ u   # position along the tilt-corrected width axis
    mid_t = (t.min() + t.max()) / 2.0

    def _widest_cross_section(sel_t, sel_s):
        if sel_t.size < 5:
            return 0.0
        lo, hi = sel_t.min(), sel_t.max()
        if hi - lo < 1e-6:
            return float(sel_s.max() - sel_s.min())
        edges = np.linspace(lo, hi, n_bins + 1)
        best = 0.0
        for i in range(n_bins):
            in_bin = (sel_t >= edges[i]) & (sel_t <= edges[i + 1])
            if in_bin.sum() >= 3:
                best = max(best, float(sel_s[in_bin].max() - sel_s[in_bin].min()))
        return best

    half_a = t < mid_t
    half_b = ~half_a
    width_a = _widest_cross_section(t[half_a], s[half_a])
    width_b = _widest_cross_section(t[half_b], s[half_b])

    mean_row_a = ys[half_a].mean() if half_a.any() else np.inf
    mean_row_b = ys[half_b].mean() if half_b.any() else np.inf
    if mean_row_a <= mean_row_b:
        top_width, bottom_width = width_a, width_b
    else:
        top_width, bottom_width = width_b, width_a

    return top_width, bottom_width, {"v": v, "u": u, "centroid": centroid, "mid_t": mid_t}

def _wider_half_on_top_elongated(main_mask):
    top_w, bottom_w, info = _half_widths_by_pca_elongated(main_mask)
    if info is None or (top_w + bottom_w) <= 1e-6:
        return None, 0.0
    margin = (top_w - bottom_w) / (top_w + bottom_w)
    return (top_w >= bottom_w), margin

def resolve_elongated_orientation(img):
    steps = []
    used_texture_fallback_step2 = False
    used_texture_fallback_step3 = False

    # --- Step 1: width > height (landscape) ---------------------------------
    mask = get_tissue_mask(img)
    h, w = _mask_bbox_hw(mask)
    if h >= w:
        img = np.ascontiguousarray(np.rot90(img, k=1))
        mask = get_tissue_mask(img)
        steps.append("rot90")
    else:
        steps.append("rot0")

    # --- Step 2: cerebellum on the right -------------------------------------
    found, landmark, area, quad = _locate_cerebellum_via_separation(mask)
    if not found:
        found, quad, landmark = _locate_cerebellum_via_texture(mask, img)
        used_texture_fallback_step2 = found

    cb_right = bool(found and quad[1] == "right")
    if found and not cb_right:
        img = np.ascontiguousarray(np.fliplr(img))
        mask = get_tissue_mask(img)
        steps.append("flipLR")
    else:
        steps.append("no_flipLR")

    # --- Step 3: main body's wider half on top -----------------------------
    labels_ws, main_id, cb_id, areas = _split_tissue_by_neck(mask)
    if labels_ws is not None and main_id is not None:
        main_mask = (labels_ws == main_id)
        top_wider, margin = _wider_half_on_top_elongated(main_mask)
    else:
        found3, quad3, _ = _locate_cerebellum_via_texture(mask, img)
        top_wider = (quad3[0] == "top") if found3 else None
        margin = 0.0
        used_texture_fallback_step3 = found3

    if top_wider is not None and not top_wider:
        img = np.ascontiguousarray(np.flipud(img))
        mask = get_tissue_mask(img)
        steps.append("flipUD")
    else:
        steps.append("no_flipUD")

    found_final, landmark_final, area_final, quad_final = _locate_cerebellum_via_separation(mask)
    if not found_final:
        found_final, quad_final, landmark_final = _locate_cerebellum_via_texture(mask, img)
    cb_found = found_final
    cb_right_final = bool(cb_found and quad_final[1] == "right")
    cb_top_final = bool(cb_found and quad_final[0] == "top")

    return {
        "final": img,
        "orientation": "+".join(steps),
        "cb_found": cb_found,
        "cb_right": cb_right_final,
        "cb_top": cb_top_final,
        "top_wider": top_wider,
        "width_margin": margin,
        "used_texture_fallback_step2": used_texture_fallback_step2,
        "used_texture_fallback_step3": used_texture_fallback_step3,
    }

def _pca_vertical_width_axes_circle(mask):
    coords = np.column_stack(np.where(mask)).astype(np.float64)  # (y, x)
    centroid = coords.mean(axis=0)
    pca = PCA(n_components=2).fit(coords)
    comps = pca.components_  # each row is a unit vector in (y, x) space
    v_idx = int(np.argmax(np.abs(comps[:, 0])))   # more "row-aligned" axis
    u_idx = 1 - v_idx
    return centroid, comps[v_idx], comps[u_idx]

def _half_widths_by_pca_circle(mask, n_bins=16):
    ys, xs = np.where(mask)
    if len(ys) < 20:
        return 0.0, 0.0, None

    coords = np.column_stack([ys, xs]).astype(np.float64)
    centroid, v, u = _pca_vertical_width_axes_circle(mask)
    rel = coords - centroid
    t = rel @ v   # position along the tilt-corrected top-to-bottom axis
    s = rel @ u   # position along the tilt-corrected width axis
    mid_t = (t.min() + t.max()) / 2.0

    def _peak_and_mean_cross_section(sel_t, sel_s):
        if sel_t.size < 5:
            return 0.0, 0.0
        lo, hi = sel_t.min(), sel_t.max()
        if hi - lo < 1e-6:
            w = float(sel_s.max() - sel_s.min())
            return w, w
        edges = np.linspace(lo, hi, n_bins + 1)
        bin_widths = []
        for i in range(n_bins):
            in_bin = (sel_t >= edges[i]) & (sel_t <= edges[i + 1])
            if in_bin.sum() >= 3:
                bin_widths.append(float(sel_s[in_bin].max() - sel_s[in_bin].min()))
        if not bin_widths:
            return 0.0, 0.0
        return max(bin_widths), float(np.mean(bin_widths))

    half_a = t < mid_t
    half_b = ~half_a
    peak_a, mean_a = _peak_and_mean_cross_section(t[half_a], s[half_a])
    peak_b, mean_b = _peak_and_mean_cross_section(t[half_b], s[half_b])

    mean_row_a = ys[half_a].mean() if half_a.any() else np.inf
    mean_row_b = ys[half_b].mean() if half_b.any() else np.inf
    if mean_row_a <= mean_row_b:
        top_width, bottom_width = peak_a, peak_b
        top_mean_width, bottom_mean_width = mean_a, mean_b
    else:
        top_width, bottom_width = peak_b, peak_a
        top_mean_width, bottom_mean_width = mean_b, mean_a

    return top_width, bottom_width, {
        "v": v, "u": u, "centroid": centroid, "mid_t": mid_t,
        "top_mean_width": top_mean_width, "bottom_mean_width": bottom_mean_width,
    }

def _wider_half_on_top_circle(mask, tie_margin_thresh=0.03):
    top_w, bottom_w, info = _half_widths_by_pca_circle(mask)
    if info is None or (top_w + bottom_w) <= 1e-6:
        return None, 0.0
    margin = (top_w - bottom_w) / (top_w + bottom_w)

    if abs(margin) < tie_margin_thresh:
        top_mean, bottom_mean = info["top_mean_width"], info["bottom_mean_width"]
        if (top_mean + bottom_mean) > 1e-6:
            mean_margin = (top_mean - bottom_mean) / (top_mean + bottom_mean)
            if abs(mean_margin) > 1e-6:
                return (top_mean >= bottom_mean), mean_margin

    return (top_w >= bottom_w), margin

def resolve_circle_orientation(img):
    steps = []

    #Step 1: landscape
    mask = get_tissue_mask(img)
    h, w = _mask_bbox_hw(mask)
    if h >= w:
        img = np.ascontiguousarray(np.rot90(img, k=1))
        mask = get_tissue_mask(img)
        steps.append("rot90")
    else:
        steps.append("rot0")

    #Step 2: circle on the right
    found, cx, cy, circle_score = _find_internal_circle(mask, img)
    if found and cx < 0.5:
        img = np.ascontiguousarray(np.fliplr(img))
        mask = get_tissue_mask(img)
        steps.append("flipLR")
    else:
        steps.append("no_flipLR")

    #Step 3: wider-half-on-top check
    top_wider, margin = _wider_half_on_top_circle(mask)
    if top_wider is None:
        top_wider, margin = True, 0.0  # no usable mask -- leave unflipped

    if not top_wider:
        img = np.ascontiguousarray(np.flipud(img))
        mask = get_tissue_mask(img)
        top_wider, margin = _wider_half_on_top_circle(mask)
        if top_wider is None:
            top_wider, margin = True, 0.0
        steps.append("flipUD")
    else:
        steps.append("no_flipUD")

    return {
        "final": img,
        "orientation": "+".join(steps),
        "circle_found": found,
        "circle_cx": cx,
        "circle_score": circle_score,
        "top_wider": top_wider,
        "width_margin": margin,
    }

def sagittal_orient_resolve(img):
    base_mask = get_tissue_mask(img)
    slice_type, aspect = classify_slice_type(base_mask)

    if slice_type == "circle":
        res = resolve_circle_orientation(img)
        return {
            "final": res["final"],
            "orientation": res["orientation"],
            "slice_type": slice_type,
            "aspect": aspect,
            "checks_passed": bool(res["circle_found"] and res["top_wider"]),
            "check1": bool(res["circle_found"]),
            "check2": bool(res["top_wider"]),
        }

    res = resolve_elongated_orientation(img)
    return {
        "final": res["final"],
        "orientation": res["orientation"],
        "slice_type": slice_type,
        "aspect": aspect,
        "checks_passed": bool(res["cb_found"] and res["cb_right"] and res["top_wider"]),
        "check1": res["cb_right"],
        "check2": bool(res["top_wider"]),
    }


In [ ]:
def compute_metrics(reoriented: np.ndarray, atlas_slice: np.ndarray,
                    ncc_margin: float, slice_type: str = "elongated") -> dict:
    if slice_type == "circle":
        pred_mask  = main_body_mask(get_tissue_mask(reoriented))
        atlas_mask = main_body_mask(get_tissue_mask(atlas_slice))
    else:
        pred_mask  = get_tissue_mask(reoriented)
        atlas_mask = get_tissue_mask(atlas_slice)

    pred_r  = (resize(pred_mask.astype(np.float32), atlas_mask.shape,
                      anti_aliasing=False, order=0) > 0.5).astype(np.uint8)
    atlas_r = atlas_mask.astype(np.uint8)

    inter   = int((pred_r & atlas_r).sum())
    dice    = (2.0 * inter) / (int(pred_r.sum()) + int(atlas_r.sum()) + 1e-8)

    try:
        hd      = hausdorff_distance(pred_r, atlas_r)
        hd_norm = hd / np.sqrt(atlas_r.shape[0]**2 + atlas_r.shape[1]**2)
    except Exception:
        hd_norm = 1.0

    ncc = _ncc_polarity_aware(reoriented, atlas_slice)

    return {
        "dice":           round(float(dice),       4),
        "hausdorff_norm": round(float(hd_norm),    4),
        "ncc":            round(float(ncc),        4),
        "ncc_margin":     round(float(ncc_margin), 4),
    }


In [ ]:
found_axes: dict[str, list[Path]] = {axis: [] for axis in ["coronal", "sagittal", "axial"]}

for axis in ["coronal", "sagittal", "axial"]:
    p = AXIS_INPUT_DIRS.get(axis)
    if p is None:
        print(f"Missing axis folder: {axis} -> (not set)")
        continue
    if p.exists():
        found_axes[axis].append(p)
    else:
        print(f"Missing axis folder: {axis} -> {p}")

print("Found axes:")
for axis, folders in found_axes.items():
    if not folders:
        print(f"  {axis:10s}  (no folders found)")
        continue
    for folder in folders:
        n = len([f for f in folder.iterdir() if f.suffix.lower() in SUPPORTED])
        print(f"  {axis:10s}  {folder}  ({n} files)")

In [ ]:
samples: dict[str, list] = {}
samples_store: dict[str, list[Path]] = {}
rng = np.random.default_rng(42)

for axis, folders in found_axes.items():
    axis_samples = []
    axis_store: list[Path] = []

    for folder in folders:
        files = [f for f in folder.iterdir() if f.suffix.lower() in SUPPORTED and is_valid_slice(f)]
        picked = random.sample(files, min(SAMPLES_PER_DATASET, len(files)))
        axis_store.extend(picked)

    rng.shuffle(axis_store)
    samples_store[axis] = axis_store

    for fpath in axis_store:
        img_clean = normalize(load_slice(fpath))
        img_rand, aug = randomize_orientation(img_clean, rng)
        pol = detect_polarity(img_rand)
        atlas_slice = find_best_atlas_slice(img_rand, axis)

        t0 = time.perf_counter()
        if axis == "sagittal":
            res = sagittal_orient_resolve(img_rand)
            final = res["final"]
            flip = f"{res['slice_type']}:{res['orientation']}" + ("" if res["checks_passed"] else " (fallback, no orientation passed both checks)")
            angle_deg = 0.0
            ncc_margin = 0.0
            atlas_slice = find_best_atlas_slice(final, axis)
            metrics = compute_metrics(final, atlas_slice, ncc_margin, slice_type=res["slice_type"])
        else:
            rotated, k, angle_deg, rot_ncc, ncc_margin = best_rotation(img_rand, atlas_slice)
            final, flip = resolve_flip(rotated, atlas_slice)
            metrics = compute_metrics(final, atlas_slice, ncc_margin)
        latency_ms = (time.perf_counter() - t0) * 1000.0

        axis_samples.append({
            "input_randomized": img_rand,
            "original_clean": img_clean,
            "reoriented": final,
            "angle_deg": angle_deg,
            "flip": flip,
            "polarity": pol,
            "filename": fpath.name,
            "metrics": metrics,
            "latency_ms": latency_ms,
            "aug": aug,
        })

        m = metrics
        print(f"  [{axis}] {fpath.name[:30]:30s}  pol={pol:+d}  "
              f"rot={angle_deg:5.0f}°  flip={str(flip):5}  "
              f"dice={m['dice']:.3f}  hd={m['hausdorff_norm']:.3f}  "
              f"ncc={m['ncc']:.3f}  margin={m['ncc_margin']:.3f}  "
              f"lat={latency_ms:.1f}ms  aug={aug}")

    samples[axis] = axis_samples
    print(f"  ── {axis}: {len(axis_samples)} done ──\n")

In [ ]:
hdr = f"{'Axis':<12} {'File':<26} {'Dice':>6} {'HD_norm':>8} {'NCC':>7} {'Margin':>8} {'Latency(ms)':>12}"
print(hdr); print("-" * len(hdr))
for axis, axis_samples in samples.items():
    for s in axis_samples:
        m = s["metrics"]
        print(f"{axis:<12} {s['filename'][:26]:<26} "
              f"{m['dice']:>6.3f} {m['hausdorff_norm']:>8.3f} "
              f"{m['ncc']:>7.3f} {m['ncc_margin']:>8.3f} {s['latency_ms']:>12.1f}")

In [ ]:
axes_list = list(samples.keys())
n_cols = 6

for axis in axes_list:
    axis_samples = samples[axis]
    n = len(axis_samples)
    n_rows = int(np.ceil(n / 3))

    fig, grid = plt.subplots(n_rows * 2, n_cols, figsize=(n_cols * 3, n_rows * 2 * 3))
    if n_rows * 2 == 1:
        grid = grid[np.newaxis, :]

    for idx, s in enumerate(axis_samples):
        row_pair = idx // 3
        col_pair = (idx % 3) * 2
        m = s["metrics"]

        grid[row_pair * 2, col_pair].imshow(s["input_randomized"], cmap="gray")
        grid[row_pair * 2, col_pair].set_title(f"RAND\n{s['filename'][:15]}", fontsize=6)
        grid[row_pair * 2, col_pair].axis("off")

        grid[row_pair * 2, col_pair + 1].imshow(s["reoriented"], cmap="gray")
        grid[row_pair * 2, col_pair + 1].set_title(
            f"REORI rot={s['angle_deg']:.0f}° fl={s['flip']}\n"
            f"dice={m['dice']:.2f} ncc={m['ncc']:.2f}", fontsize=6)
        grid[row_pair * 2, col_pair + 1].axis("off")

    for idx in range(len(axis_samples), n_rows * 3):
        row_pair = idx // 3
        col_pair = (idx % 3) * 2
        grid[row_pair * 2, col_pair].axis("off")
        grid[row_pair * 2, col_pair + 1].axis("off")

    for col in range(n_cols):
        for row in range(1, n_rows * 2, 2):
            grid[row, col].axis("off")

    plt.suptitle(f"{axis} — Randomized vs Reoriented", fontsize=11)
    plt.tight_layout()
    plt.show()